In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
from collections import Counter
import random
from tqdm import tqdm
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
import time
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

**Loading the Dataset**

This cell loads the dataset from a pickle file named `train.csv`, which contains the training data for title generation. Each example is a dictionary with 'article' and 'title' keys.


In [ ]:
start_time = time.time()
train_path = "/content/drive/MyDrive/Dataset/train.csv"
test_path = "/content/drive/MyDrive/Dataset/test.csv"

print("Loading datasets...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Datasets loaded successfully!")

end_time = time.time()
print(f"load dataset time: {end_time - start_time:.2f} seconds")

Loading datasets...
Datasets loaded successfully!
load dataset time: 8.29 seconds


###  Splitting the Training Data into Train and Validation Sets

This cell extracts 500 random samples from the `train_df` DataFrame to create a **validation set**, which will be used later to evaluate the model during training. The `random_state=42` ensures reproducibility of the split. After sampling, the selected rows are removed from the original training set (`train_df`) to prevent data leakage. The final sizes of both the updated training set and the new validation set are then printed.


In [ ]:
start_time = time.time()
print("\nExtracting 500 articles for validation set...")
validation_df = train_df.sample(n=500, random_state=42)
train_df = train_df.drop(validation_df.index)

print(f"\nTrain Set Size After Splitting: {train_df.shape[0]} articles")
print(f"Validation Set Size: {validation_df.shape[0]} articles")
end_time = time.time()
print(f"Load Dataset time: {end_time - start_time:.2f} seconds")


Extracting 500 articles for validation set...

Train Set Size After Splitting: 13379 articles
Validation Set Size: 500 articles
Load Dataset time: 0.01 seconds


###  Exploring Dataset Columns and Checking for Missing Values

This cell performs an initial inspection of the datasets:
- Displays the column names for the training and test datasets.
- Checks for **missing (null) values** in the training, validation, and test datasets using `isnull().sum()`.
- Optionally displays a few sample rows from the training set that contain missing values, if any.
This step helps ensure the datasets are clean and identifies any necessary preprocessing steps related to missing data.


In [ ]:
# Display column names
print(" Columns in Train Dataset:")
print(train_df.columns.tolist())

print("\n Columns in Test Dataset:")
print(test_df.columns.tolist())

# Check for missing values
print("\n Checking for missing/null values in Train Dataset:")
print(train_df.isnull().sum())

print("\n Checking for missing/null values in Validation Dataset:")
print(validation_df.isnull().sum())

print("\n Checking for missing/null values in Test Dataset:")
print(test_df.isnull().sum())

print("\n Sample rows with missing values in train:")
print(train_df[train_df.isnull().any(axis=1)].head())

print("\n Column info and missing value check completed.")

 Columns in Train Dataset:
['title', 'text']

 Columns in Test Dataset:
['title', 'text']

 Checking for missing/null values in Train Dataset:
title    0
text     0
dtype: int64

 Checking for missing/null values in Validation Dataset:
title    0
text     0
dtype: int64

 Checking for missing/null values in Test Dataset:
title    0
text     0
dtype: int64

 Sample rows with missing values in train:
Empty DataFrame
Columns: [title, text]
Index: []

 Column info and missing value check completed.


###  Initializing Text Preprocessing Tools

This cell initializes tools for text preprocessing:
- Creates an instance of **`WordNetLemmatizer`**, which will be used to reduce words to their base (lemma) form.
- Loads a set of **English stopwords** from NLTK, which are common words (like "is", "the", "and") that are usually removed in text processing to reduce noise.
Note: The lemmatizer is initialized twice, which is redundant and can be safely removed without affecting functionality.


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

###  Text Preprocessing Function

This function `preprocess_text(text)` performs a series of cleaning and normalization steps on raw input text:
1. **Missing Value Handling**: If the input is `NaN`, it returns an empty string.
2. **Lowercasing**: Converts all characters to lowercase to maintain consistency.
3. **Non-ASCII Removal**: Replaces newline characters and removes non-ASCII characters to clean the text.
4. **Punctuation Removal**: Uses `str.translate` to eliminate punctuation symbols from the text.
5. **Tokenization**: Splits the text into individual word tokens using `nltk.word_tokenize`.
6. **Stopword Removal**: Filters out common stopwords using the previously loaded list.
7. **Lemmatization**: Reduces words to their base form using the `WordNetLemmatizer`.

The function returns the cleaned and normalized string, ready for further NLP tasks.


In [ ]:
def preprocess_text(text):
    if pd.isnull(text):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove non-ASCII characters
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword removal
    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

###  Combining Title and Text + Preprocessing

This cell performs two major operations:

1. **Combining Title and Text**:
   - It concatenates the `title` and `text` columns into a new column called `combined_text` for the `train_df`, `validation_df`, and `test_df`.
   - This combination ensures that the model can utilize both the article title and its full content during training and evaluation.

2. **Text Preprocessing**:
   - The `combined_text` column in each dataset is processed using the `preprocess_text()` function defined earlier.
   - This cleans and normalizes the text for better performance during modeling.

After processing, the text in all datasets is in a clean and standardized format, suitable for tokenization and training downstream NLP models.


In [ ]:
# Combine title and text into one column
start_time = time.time()
train_df["combined_text"] = train_df["title"] + " " + train_df["text"]
validation_df["combined_text"] = validation_df["title"] + " " + validation_df["text"]
test_df["combined_text"] = test_df["title"] + " " + test_df["text"]

text_column = "combined_text"

print(f" Preprocessing '{text_column}' column in datasets...")

train_df[text_column] = train_df[text_column].apply(preprocess_text)
validation_df[text_column] = validation_df[text_column].apply(preprocess_text)
test_df[text_column] = test_df[text_column].apply(preprocess_text)

print(" All datasets have been preprocessed successfully.")
end_time = time.time()
print(f"Preprocessing: {end_time - start_time:.2f} seconds")


 Preprocessing 'combined_text' column in datasets...
 All datasets have been preprocessed successfully.
Preprocessing: 193.90 seconds


###  Saving Preprocessed Datasets

This cell handles the saving of the cleaned and preprocessed datasets:

1. **File Paths Setup**:
   - It defines the file paths where the preprocessed training, validation, and test sets will be stored.

2. **Saving CSVs**:
   - Each of the DataFrames (`train_df`, `validation_df`, and `test_df`) is saved as a CSV file without the index column.

3. **Logging**:
   - Informative print statements confirm the success of the preprocessing and indicate where the files are saved.

This step ensures that the cleaned datasets can be reloaded later without re-running the entire preprocessing pipeline, improving efficiency for experimentation and training.


In [ ]:
print("Text preprocessing function defined successfully!")

preprocessed_train_path = "/content/preprocessed_train.csv"
preprocessed_validation_path = "/content/preprocessed_validation.csv"
preprocessed_test_path = "/content/preprocessed_test.csv"

print("\nSaving preprocessed datasets...")

# Save preprocessed datasets
train_df.to_csv(preprocessed_train_path, index=False)
validation_df.to_csv(preprocessed_validation_path, index=False)
test_df.to_csv(preprocessed_test_path, index=False)

print("\nPreprocessing completed successfully! Files saved at:")
print(preprocessed_train_path)
print(preprocessed_validation_path)
print(preprocessed_test_path)


Text preprocessing function defined successfully!

Saving preprocessed datasets...

Preprocessing completed successfully! Files saved at:
/content/preprocessed_train.csv
/content/preprocessed_validation.csv
/content/preprocessed_test.csv
